# Stage 2 — Exhibit 99.x Fetcher

Every filing in `filings_raw.csv` is a stub pointing to an exhibit.  
This notebook fetches the actual earnings press release text (Exhibit 99.1, 99, or 99.2) for all 6,131 filings.

**Input** : `data/filings_raw.csv`  
**Output** : `data/filings_with_text.csv` — same structure, `bodyText` replaced with real exhibit content  
**Runtime** : ~25–35 minutes (one EDGAR request per filing)

## 0. Imports and configuration

In [ ]:
import requests
import pandas as pd
import json
import time
import re
import logging
import warnings
from pathlib import Path
from bs4 import BeautifulSoup, XMLParsedAsHTMLWarning
from tqdm.notebook import tqdm

warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

# ── Update this ───────────────────────────────────────────────────────────────
EDGAR_USER_AGENT = "Jargalsaikhan Gansuld jargalsaikha_gansuld@student.ceu.edu"

# ── Paths ─────────────────────────────────────────────────────────────────────
DATA_DIR         = Path("data")
INPUT_PATH       = DATA_DIR / "filings_raw.csv"
OUTPUT_PATH      = DATA_DIR / "filings_with_text.csv"
CHECKPOINT_FILE  = DATA_DIR / "checkpoint_stage2.json"
DATA_DIR.mkdir(exist_ok=True)

# ── Rate limiting ─────────────────────────────────────────────────────────────
REQUEST_DELAY = 0.12
MAX_RETRIES   = 3

# ── Logging ───────────────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)s  %(message)s",
    handlers=[
        logging.FileHandler(DATA_DIR / "stage2.log"),
        logging.StreamHandler(),
    ],
)
log = logging.getLogger(__name__)
HEADERS = {"User-Agent": EDGAR_USER_AGENT}

print("Configuration loaded.")

## 1. Load Stage 1 output

In [ ]:
df = pd.read_csv(INPUT_PATH)
print(f"Loaded {len(df)} filings from {INPUT_PATH}")
print(f"Columns: {list(df.columns)}")

## 2. Helper — HTTP with retries

In [ ]:
def _get(url: str, retries: int = MAX_RETRIES) -> requests.Response:
    """GET with automatic retry on 5xx and rate limit errors."""
    for attempt in range(retries):
        try:
            resp = requests.get(url, headers=HEADERS, timeout=15)
            if resp.status_code == 200:
                time.sleep(REQUEST_DELAY)
                return resp
            elif resp.status_code == 429:
                log.warning("Rate limited — sleeping 60s...")
                time.sleep(60)
            elif resp.status_code >= 500:
                log.warning(f"Server error {resp.status_code}, retry {attempt + 1}")
                time.sleep(2 ** attempt)
            else:
                resp.raise_for_status()
        except requests.exceptions.RequestException as e:
            log.warning(f"Request failed ({e}), retry {attempt + 1}")
            time.sleep(2 ** attempt)
    raise RuntimeError(f"Failed to fetch {url} after {retries} retries")

## 3. Fetch filing index and find Exhibit 99.x

In [ ]:
def get_exhibit_url(cik: str, accession: str) -> str | None:
    """
    Fetches the EDGAR filing index and finds the best EX-99* document.
    Prefers primary press releases over non-GAAP supplements or slide decks.
    """
    acc_nodash = accession.replace("-", "")
    index_url  = f"https://www.sec.gov/Archives/edgar/data/{int(cik)}/{acc_nodash}/{accession}-index.htm"

    try:
        resp = _get(index_url)
    except Exception as e:
        log.warning(f"Index fetch failed for {accession}: {e}")
        return None

    soup = BeautifulSoup(resp.text, "lxml")

    candidates = []
    for row in soup.find_all("tr"):
        cells = row.find_all("td")
        if len(cells) < 4:
            continue
        doc_type = cells[3].get_text(strip=True).upper()
        if doc_type.startswith("EX-99"):
            link = cells[2].find("a")
            if link and link.get("href"):
                href = link["href"]
                full_url = href if href.startswith("http") else "https://www.sec.gov" + href
                candidates.append(full_url)

    if not candidates:
        return None

    SKIP_KEYWORDS = ["nongaap", "non-gaap", "supplement", "slides", "presentation", "tables", "nonxgaap"]

    for url in candidates:
        filename = url.split("/")[-1].lower()
        if not any(kw in filename for kw in SKIP_KEYWORDS):
            return url

    return candidates[0]

## 4. Download and clean exhibit text

In [ ]:
def fetch_and_clean_exhibit(url: str) -> str | None:
    """
    Downloads an exhibit and returns raw plain text.
    Only removes non-text HTML elements (scripts, styles, tables) at parse time.
    All further cleaning is handled in the preprocessing notebook.
    """
    try:
        resp = _get(url)
    except Exception as e:
        log.warning(f"Exhibit fetch failed for {url}: {e}")
        return None

    content_type = resp.headers.get("Content-Type", "")

    if "html" in content_type or url.lower().endswith((".htm", ".html")):
        soup = BeautifulSoup(resp.text, "lxml")
        for tag in soup(["script", "style", "head", "table"]):
            tag.decompose()
        text = soup.get_text(separator=" ", strip=True)
    else:
        text = resp.text

    return text if text and len(text.split()) >= 50 else None

## 5. Checkpoint helpers

In [ ]:
def load_checkpoint() -> set[str]:
    """Returns set of accession numbers already processed."""
    if CHECKPOINT_FILE.exists():
        with open(CHECKPOINT_FILE) as f:
            return set(json.load(f).get("completed", []))
    return set()


def save_checkpoint(completed: set[str]):
    with open(CHECKPOINT_FILE, "w") as f:
        json.dump({"completed": list(completed)}, f)


def save_results(rows: list[dict]):
    if not rows:
        return
    pd.DataFrame(rows).to_csv(OUTPUT_PATH, index=False)
    log.info(f"Saved {len(rows)} rows → {OUTPUT_PATH}")

## 6. Check resume state

In [ ]:
completed = load_checkpoint()

if OUTPUT_PATH.exists():
    rows = pd.read_csv(OUTPUT_PATH).to_dict("records")
    print(f"Resuming — {len(rows)} filings already processed.")
else:
    rows = []

remaining = len(df) - len(completed)
print(f"Already processed : {len(completed)}")
print(f"Remaining         : {remaining}")
print(f"Estimated runtime : ~{remaining * REQUEST_DELAY / 60:.0f} minutes")

In [ ]:
if CHECKPOINT_FILE.exists():
    CHECKPOINT_FILE.unlink()
    print("Checkpoint cleared")

if OUTPUT_PATH.exists():
    OUTPUT_PATH.unlink()
    print("Output file cleared")

rows = []
completed = set()
print(f"rows: {len(rows)}")
print(f"completed: {len(completed)}")
print("Ready to run.")

## 7. Run the pipeline

In [ ]:
failed = []   # track filings where no exhibit was found

for _, filing in tqdm(df.iterrows(), total=len(df), desc="Filings", unit="filing"):

    acc = filing["accessionNumber"]

    if acc in completed:
        continue

    cik = str(filing["cik"]).zfill(10)

    # Step 1: get exhibit URL from filing index
    exhibit_url = get_exhibit_url(cik, acc)

    if exhibit_url is None:
        log.warning(f"No EX-99* found: {filing['ticker']} {acc}")
        failed.append(acc)
        completed.add(acc)
        save_checkpoint(completed)
        continue

    # Step 2: download and clean exhibit text
    exhibit_text = fetch_and_clean_exhibit(exhibit_url)

    if exhibit_text is None or len(exhibit_text.split()) < 50:
        log.warning(f"Empty exhibit: {filing['ticker']} {acc}")
        failed.append(acc)
        completed.add(acc)
        save_checkpoint(completed)
        continue

    rows.append({
        "ticker":          filing["ticker"],
        "cik":             filing["cik"],
        "accessionNumber": acc,
        "filingDate":      filing["filingDate"],
        "filingDatetime":  filing["filingDatetime"],
        "primaryDocument": filing["primaryDocument"],
        "exhibitUrl":      exhibit_url,
        "bodyText":        exhibit_text,
        "bodyTextLen":     len(exhibit_text),
    })

    completed.add(acc)
    save_checkpoint(completed)

    # Incremental save every 100 filings
    if len(completed) % 100 == 0:
        save_results(rows)

# Final save
save_results(rows)
print(f"\nDone.")
print(f"Successful : {len(rows)}")
print(f"Failed     : {len(failed)}")

In [ ]:
print(f"Rows saved: {len(rows)}")
print(f"Failed: {len(failed)}")
print(f"Completed set size: {len(completed)}")

## 8. Validation

In [ ]:
df_out = pd.read_csv(OUTPUT_PATH)

print("=" * 50)
print(f"Total filings        : {len(df_out)}")
print(f"Unique companies     : {df_out['ticker'].nunique()}")
print(f"Median body text len : {df_out['bodyTextLen'].median():.0f} chars")
print(f"Min body text len    : {df_out['bodyTextLen'].min()} chars")
print(f"Max body text len    : {df_out['bodyTextLen'].max()} chars")
print(f"\nText length distribution:")
print(df_out['bodyTextLen'].describe().round(0))

In [ ]:
df2 = pd.read_csv("data/filings_with_text.csv")
df1 = pd.read_csv("data/filings_raw.csv")

# Filings in Stage 1 but not in Stage 2
missing_accs = set(df1['accessionNumber']) - set(df2['accessionNumber'])
missing_df = df1[df1['accessionNumber'].isin(missing_accs)]
print(f"Filings dropped in Stage 2: {len(missing_df)}")
print("\nWatchlist firms affected:")
print(missing_df[missing_df['ticker'].isin(['SIVB','TWTR','SBNY'])][['ticker','filingDate']])

In [ ]:
import pandas as pd
df = pd.read_csv("data/filings_with_text.csv")
print(df.nlargest(5, 'bodyTextLen')[['ticker', 'filingDate', 'exhibitUrl', 'bodyTextLen']])

In [ ]:
stz = df[df['bodyTextLen'] == df['bodyTextLen'].max()].iloc[0]
print(stz['exhibitUrl'])
print(stz['bodyText'][:500])

In [ ]:
import pandas as pd
df = pd.read_csv("data/filings_with_text.csv")
before = len(df)
df = df[~df['exhibitUrl'].str.lower().str.endswith('.pdf')]
after = len(df)
print(f"Removed {before - after} PDF filings")
print(f"Remaining: {after}")
df.to_csv("data/filings_with_text.csv", index=False)

In [ ]:
df_out = pd.read_csv(OUTPUT_PATH)

print("=" * 50)
print(f"Total filings        : {len(df_out)}")
print(f"Unique companies     : {df_out['ticker'].nunique()}")
print(f"Median body text len : {df_out['bodyTextLen'].median():.0f} chars")
print(f"Min body text len    : {df_out['bodyTextLen'].min()} chars")
print(f"Max body text len    : {df_out['bodyTextLen'].max()} chars")
print(f"\nText length distribution:")
print(df_out['bodyTextLen'].describe().round(0))

In [ ]:
print(df.nsmallest(5, 'bodyTextLen')[['ticker', 'filingDate', 'exhibitUrl', 'bodyTextLen']])

In [ ]:
print(df[df['ticker'] == 'SRE'].iloc[0]['bodyText'])

In [ ]:
df = pd.read_csv("data/filings_with_text.csv")
shortest = df.nsmallest(1, 'bodyTextLen').iloc[0]
print(shortest['ticker'], shortest['filingDate'], shortest['bodyTextLen'])
print(shortest['bodyText'])

In [ ]:
df = pd.read_csv("data/filings_with_text.csv")

# Check how many have EX-99.2 style URLs
mask = df['exhibitUrl'].str.lower().str.contains('ex99_2|ex99-2|ex992')
print(f"EX-99.2 filings: {mask.sum()}")
print(df[mask][['ticker', 'filingDate', 'bodyTextLen', 'exhibitUrl']].head(10))

In [ ]:
df = pd.read_csv("data/filings_with_text.csv")
before = len(df)
df = df[df['bodyTextLen'] >= 1000]
print(f"Removed {before - len(df)} filings")
print(f"Remaining: {len(df)}")
print(f"New min: {df['bodyTextLen'].min()}")
df.to_csv("data/filings_with_text.csv", index=False)